# model 1

In [20]:
from keras.layers import Input, UpSampling2D
from keras.applications.resnet50 import ResNet50
from keras.models import Model
from keras import backend as K

K.set_learning_phase(0)

# 32x32 입력을 받아서 224x224(7배)로 확대
input_tensor = Input(shape=(32, 32, 3))
resizing_layer = UpSampling2D(size=(7, 7))(input_tensor) # 32 * 7 = 224

# 확대된 텐서를 ResNet50에 전달
base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=resizing_layer)

# model 2

In [21]:
base_model_2 = ResNet50(weights=None, include_top=False, input_tensor=resizing_layer)

model_list = [base_model, base_model_2]

# data

In [22]:
import os
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import cifar10
from keras.utils import to_categorical

# 1. 결과물 저장 폴더 위치 지정 

results_dir = "C:/Users/USER/Desktop/results"
print("결과물 저장 위치가 '{}' 폴더로 지정되었습니다.".format(results_dir))

# 2. CIFAR-10 데이터 로드 
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 전처리: 0~1 사이로 정규화 및 데이터 형식 맞춤 
x_test = x_test.astype('float32') / 255.0
y_test_cat = to_categorical(y_test, 10)

print("데이터 로드 및 전처리 완료. 테스트 데이터 크기:", x_test.shape)

결과물 저장 위치가 'C:/Users/USER/Desktop/results' 폴더로 지정되었습니다.
데이터 로드 및 전처리 완료. 테스트 데이터 크기: (10000, 32, 32, 3)


In [23]:
from keras import backend as K
import os
import numpy as np

# 1. 예측용 함수 정의 (TypeError 방지를 위해 명시적 구조 사용)
# K.learning_phase() 텐서에 대응하는 값을 함께 입력받도록 설정합니다.
predict_fn1 = K.function([base_model.input, K.learning_phase()], [base_model.output])
predict_fn2 = K.function([base_model_2.input, K.learning_phase()], [base_model_2.output])

def calculate_neuron_coverage(model, input_data, threshold=0.1):
    """뉴런 커버리지 계산: 임계값 이상 활성화된 뉴런 비율 [cite: 27, 43, 73]"""
    layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name or 'dense' in layer.name]
    get_activations = K.function([model.input, K.learning_phase()], layer_outputs)
    
    # 0(추론 모드)을 리스트 인자로 전달
    activations = get_activations([input_data, 0])
    
    total_neurons = 0
    activated_neurons = 0
    for layer_act in activations:
        total_neurons += layer_act.size
        activated_neurons += np.count_nonzero(layer_act > threshold)
    
    return (activated_neurons / float(total_neurons)) * 100

# 2. 불일치(Disagreement) 사례 탐색 (최소 5개) [cite: 40, 42, 45, 77]
disagreement_inputs = []
print("불일치 사례를 찾는 중...")

# 처음 200개 이미지 중 탐색 (불일치가 적을 수 있어 범위를 늘림)
for i in range(200): 
    img = x_test[i:i+1]
    
    # predict_fn을 사용하여 TypeError 해결
    out1 = predict_fn1([img, 0])[0]
    out2 = predict_fn2([img, 0])[0]
    
    pred1 = np.argmax(out1)
    pred2 = np.argmax(out2)
    
    if pred1 != pred2:
        disagreement_inputs.append(x_test[i])
        
        # 이미지 시각화 및 지정된 results_dir에 저장 [cite: 45, 77, 93]
        plt.imshow(x_test[i])
        plt.title("M1: %d | M2: %d" % (pred1, pred2))
        plt.axis('off')
        
        save_name = 'disagreement_%d.png' % len(disagreement_inputs)
        plt.savefig(os.path.join(results_dir, save_name))
        plt.close()
        
        if len(disagreement_inputs) >= 5:
            break

# 3. 최종 결과 출력 [cite: 72, 73]
nc_model1 = calculate_neuron_coverage(base_model, x_test[:10])
nc_model2 = calculate_neuron_coverage(base_model_2, x_test[:10])

print("\n" + "="*30)
print("발견된 불일치 입력 개수: %d" % len(disagreement_inputs))
print("Model 1 뉴런 커버리지: %.2f%%" % nc_model1)
print("Model 2 뉴런 커버리지: %.2f%%" % nc_model2) 
print("결과 저장 경로: %s" % results_dir)
print("="*30)

불일치 사례를 찾는 중...


TypeError: Cannot interpret feed_dict key as Tensor: Can not convert a int into a Tensor.